# Exercice 1.5 bis **amélioré** : Feature engineering — localisation + caractéristiques du bien

Comme **E5bis**, avec en plus des variables **numériques** liées à la taille et à la qualité :
- `OverallQual` (qualité globale), `GrLivArea`, `GarageArea`, `TotalBsmtSF`.
**y** = `SalePrice`. Même principes : split, `Pipeline`, pas de fuite de données.


## Règle : pas de fuite de données

`fit` sur le train uniquement pour toute transformation.


## Section 1: Imports, chargement, split


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import root_mean_squared_error

df = pd.read_csv("AmesHousing.csv")
df.columns = df.columns.str.replace(" ", "")

COLS_LOC = ["Neighborhood", "MSZoning", "Condition1", "Condition2"]
COLS_NUM = ["OverallQual", "GrLivArea", "GarageArea", "TotalBsmtSF"]
FEATURES = COLS_LOC + COLS_NUM

X = df[FEATURES]
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=666
)
print("X_train.shape", X_train.shape, "| X_test.shape", X_test.shape)

cat_cols = COLS_LOC
num_cols = COLS_NUM

num_pipe = Pipeline(
    [
        ("imp", SimpleImputer(strategy="median")),
        ("sc", RobustScaler()),
    ]
)
cat_pipe = Pipeline(
    [
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

def make_ohe_prep():
    return ColumnTransformer(
        [
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ]
    )



X_train.shape (1537, 8) | X_test.shape (660, 8)


## Section 2: Encodage (OHE pour les nominaux)


In [3]:
base = Pipeline(
    [
        ("prep", make_ohe_prep()),
        ("model", Lasso(alpha=1.0, max_iter=20000)),
    ]
)
base.fit(X_train, y_train)
pred_b = base.predict(X_test)
rmse_b = root_mean_squared_error(y_test, pred_b)
print("RMSE baseline (OHE + Lasso) :", round(rmse_b, 2))


RMSE baseline (OHE + Lasso) : 34960.56


## Section 3: Cible log + même pipeline


In [4]:
y_tr_log = np.log1p(y_train)

base_log = Pipeline(
    [
        ("prep", make_ohe_prep()),
        ("model", Ridge(alpha=10.0)),
    ]
)
base_log.fit(X_train, y_tr_log)
pred_log = np.expm1(base_log.predict(X_test))
rmse_log = root_mean_squared_error(y_test, pred_log)
print("RMSE avec log1p sur y + Ridge :", round(rmse_log, 2))


RMSE avec log1p sur y + Ridge : 33602.35


## Section 4: Modèle enrichi — Target Encoding sur `Neighborhood` + OHE sur le reste


In [5]:
te_nbh = TargetEncoder(random_state=42, target_type="continuous")

num_pipe_fe = Pipeline(
    [
        ("imp", SimpleImputer(strategy="median")),
        ("sc", RobustScaler()),
    ]
)

transformers = [
    ("te_nbh", te_nbh, ["Neighborhood"]),
    (
        "ohe_rest",
        Pipeline(
            [
                ("imp", SimpleImputer(strategy="most_frequent")),
                (
                    "ohe",
                    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                ),
            ]
        ),
        ["MSZoning", "Condition1", "Condition2"],
    ),
    ("num", num_pipe_fe, num_cols),
]

prep_fe = ColumnTransformer(transformers)

pipe_fe = Pipeline(
    [
        ("prep", prep_fe),
        ("model", Ridge(alpha=10.0)),
    ]
)

pipe_fe.fit(X_train, y_tr_log)
pred_fe = np.expm1(pipe_fe.predict(X_test))
rmse_fe = root_mean_squared_error(y_test, pred_fe)
print("RMSE pipeline FE (TE Neighborhood + OHE + num + log y) :", round(rmse_fe, 2))
print(
    "Comparaison RMSE ($) : baseline Lasso",
    round(rmse_b, 2),
    "| Ridge+log y",
    round(rmse_log, 2),
)



RMSE pipeline FE (TE Neighborhood + OHE + num + log y) : 33305.14
Comparaison RMSE ($) : baseline Lasso 34960.56 | Ridge+log y 33602.35


## Synthèse

| Modèle | RMSE ($) |
|--------|----------|
| Lasso + num + OHE (y en dollars) | *voir sortie* |
| Ridge + num + OHE + `log1p(y)` | *voir sortie* |
| Ridge + TE `Neighborhood` + OHE + num + `log1p(y)` | *voir sortie* |

Les variables **surface / qualité** réduisent fortement la RMSE par rapport au seul jeu « localisation ».
